In [1]:
import re
import torch
import torchvision.transforms as transforms
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch import nn
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt


max_epoch = 25
BATCH_SIZE = 256
LR = 0.001
log_interval = 10
val_interval = 1
classes = 2
start_epoch = -1
lr_decay_step = 7
print_interval = 2
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
log_dir = os.path.abspath('./logs')

class CatDataset(Dataset):
    def __init__(self, image_path, transform=None, is_train=True):
        self.paths = []
        self.labels = []
        for w, _, files in os.walk(image_path):
            for f in files:
                if f.endswith('.jpg') or f.endswith('.png') or f.endswith('.jpeg'):
                    self.paths.append(os.path.join(w, f))
                    # 1是有耄耋面相，0不是耄耋
                    self.labels.append(int(re.findall(r'\d+', f.split('-')[-1])[0]))
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        # 读取图像
        image = Image.open(self.paths[idx]).convert('RGB')

        # 应用数据增强
        if self.transform:
            image = self.transform(image)

        return image, self.labels[idx]


train_transform = transforms.Compose([
    # 调整图像大小
    transforms.Resize((224, 224)),
    # 随机水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 随机旋转左右90度
    transforms.RandomRotation(degrees=90),
    # 随机调节亮度、对比度、饱和度和色相
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),

    # 三通道标准化，来自imagenet
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [2]:
draw = False

image_path = './dataset/maodie'

train_dataset = CatDataset(
    image_path=image_path,
    transform=transforms.Compose(train_transform.transforms[:-1]) if draw else train_transform,
    is_train=True
)

val_dataset = CatDataset(
    image_path=image_path,
    transform=val_transform,
    is_train=False
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

if draw:
    for images, labels in train_loader:
        img = np.asarray(images[0])
        plt.imshow(img.transpose((1, 2, 0)))
        plt.show()
        break

In [5]:
from yu_utils import show_conf_mat
from torchvision.models import resnet50
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('./logs', filename_suffix="_finetune_res50")
net = resnet50(pretrained=True)
# 冻结底部卷积层
for param in net.parameters():
    param.requires_grad = False

in_features = net.fc.in_features
net.fc = nn.Linear(in_features, classes)
net = net.to(device)


loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9)
# 每7个epoch学习率*0.1
scheduler = lr_scheduler.StepLR(optimizer, step_size=lr_decay_step, gamma=0.1)


for epoch in range(start_epoch + 1, max_epoch):

    class_num = classes
    conf_mat = np.zeros((class_num, class_num))
    loss_sigma = []
    loss_avg = 0
    acc_avg = 0
    path_error = []
    label_list = []

    net.train()
    for i, data in enumerate(train_loader):

        # forward
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # forward, backward, update weights
        optimizer.zero_grad()
        outputs = net(inputs)
        l = loss(outputs, labels)
        l.backward()
        optimizer.step()

        # 统计分类情况
        # 统计预测信息
        loss_sigma.append(l.item())
        loss_avg = np.mean(loss_sigma)

        _, predicted = torch.max(outputs.data, 1)
        for j in range(len(labels)):
            cate_i = labels[j].cpu().numpy()
            pre_i = predicted[j].cpu().numpy()
            conf_mat[cate_i, pre_i] += 1.
        acc_avg = conf_mat.trace() / conf_mat.sum()

        # 打印训练信息
        # 每10个iteration 打印一次训练信息，loss为10个iteration的平均
        if i % print_interval == print_interval - 1:
            print("Training: Epoch[{:0>3}/{:0>3}] Iteration[{:0>3}/{:0>3}] Loss: {:.4f} Acc:{:.2%}".
                  format(epoch + 1, max_epoch, i + 1, len(train_loader), loss_avg, acc_avg))

            print("epoch:{} conv1.weights[0, 0, ...] :\n {}".format(epoch, net.conv1.weight[0, 0, ...]))

    scheduler.step()  # 更新学习率
    # 记录训练loss
    writer.add_scalars('Loss_group', {'train_loss': loss_avg}, epoch)
    # 记录learning rate
    writer.add_scalar('learning rate', scheduler.get_last_lr()[0], epoch)
    # 记录Accuracy
    writer.add_scalars('Accuracy_group', {'train_acc': acc_avg}, epoch)

    conf_mat_figure = show_conf_mat(conf_mat, ['maodie', 'not maodie'] , "train", log_dir, epoch=epoch, verbose=epoch == max_epoch - 1)
    writer.add_figure('confusion_matrix_train', conf_mat_figure, global_step=epoch)

    # validate the model
    class_num = classes
    conf_mat = np.zeros((class_num, class_num))
    loss_sigma = []
    loss_avg = 0
    acc_avg = 0
    path_error = []
    label_list = []

    net.eval()
    with torch.no_grad():
        for j, data in enumerate(val_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = net(inputs)
            l = loss(outputs, labels)

            # 统计预测信息
            loss_sigma.append(l.item())
            loss_avg = np.mean(loss_sigma)

            _, predicted = torch.max(outputs.data, 1)
            for j in range(len(labels)):
                cate_i = labels[j].cpu().numpy()
                pre_i = predicted[j].cpu().numpy()
                conf_mat[cate_i, pre_i] += 1.
            acc_avg = conf_mat.trace() / conf_mat.sum()
    print('{} set Accuracy:{:.2%}'.format('Valid', conf_mat.trace() / conf_mat.sum()))
    # 记录Loss, accuracy
    writer.add_scalars('Loss_group', {'valid_loss': loss_avg}, epoch)
    writer.add_scalars('Accuracy_group', {'valid_acc': acc_avg}, epoch)
    # 保存混淆矩阵图
    conf_mat_figure = show_conf_mat(conf_mat, ['maodie', 'not maodie'], "valid", log_dir, epoch=epoch, verbose=epoch == max_epoch - 1)
    writer.add_figure('confusion_matrix_valid', conf_mat_figure, global_step=epoch)
print('Finished Training')

Valid set Accuracy:47.22%
Valid set Accuracy:50.00%
Valid set Accuracy:63.89%
Valid set Accuracy:47.22%
Valid set Accuracy:52.78%
Valid set Accuracy:58.33%
Valid set Accuracy:66.67%
Valid set Accuracy:69.44%
Valid set Accuracy:69.44%
Valid set Accuracy:69.44%
Valid set Accuracy:75.00%
Valid set Accuracy:75.00%
Valid set Accuracy:77.78%
Valid set Accuracy:77.78%
Valid set Accuracy:77.78%
Valid set Accuracy:77.78%
Valid set Accuracy:77.78%
Valid set Accuracy:77.78%
Valid set Accuracy:80.56%
Valid set Accuracy:80.56%
Valid set Accuracy:80.56%
Valid set Accuracy:83.33%
Valid set Accuracy:83.33%
Valid set Accuracy:80.56%
class:maodie    , total num:19.0  , correct num:19.0   Recall: 100.00% Precision: 82.61%
class:not maodie, total num:17.0  , correct num:13.0   Recall: 76.47% Precision: 100.00%
Valid set Accuracy:80.56%
class:maodie    , total num:19.0  , correct num:18.0   Recall: 94.74% Precision: 75.00%
class:not maodie, total num:17.0  , correct num:11.0   Recall: 64.71% Precision: 91.